# HDWI Kernels on a T4

Runs the GPU tests and the kernel benchmark from
[docs/gpu-performance.md](../docs/gpu-performance.md) on a free Kaggle
T4.

Run on Kaggle with **Accelerator: GPU T4 x2** and **Internet: on**. The
notebook uses one GPU. It gets the project from GitHub. To use a local
copy instead, upload the repository as a Kaggle dataset named
`earth-intelligence` and add it to the notebook.

Output: `/kaggle/working/gpu_perf.zip`.

## Environment

PyTorch builds the CUDA kernel with `nvcc`, so the `nvcc` release
should match PyTorch's CUDA version. The Triton kernel needs a Triton
release that supports compute capability 7.5.

In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
import shutil
import subprocess
from pathlib import Path

import torch
import triton

print("torch", torch.__version__, "cuda", torch.version.cuda)
print("triton", triton.__version__)
print("compute capability", torch.cuda.get_device_capability(0))

## Project

In [ ]:
REPOSITORY = "https://github.com/khoahonguyen0210/Earth_intelligence.git"
INPUT = Path("/kaggle/input")
PROJECT = Path("/kaggle/working/earth-intelligence")
OUTPUT = Path("/kaggle/working/gpu_perf")


def uploaded_project():
    for marker in INPUT.rglob("ml/pyproject.toml"):
        return marker.parent.parent
    for archive in INPUT.rglob("earth-intelligence.tar.gz"):
        shutil.unpack_archive(archive, PROJECT)
        return PROJECT
    return None


if not PROJECT.exists():
    source = uploaded_project()
    if source is None:
        subprocess.run(
            ["git", "clone", "--depth", "1", REPOSITORY, str(PROJECT)], check=True
        )
    elif source != PROJECT:
        shutil.copytree(source, PROJECT)
OUTPUT.mkdir(parents=True, exist_ok=True)

In [ ]:
!pip install -q ninja pytest cdsapi python-dotenv

## Tests

All four versions against the float64 reference: edge cases, random
columns and surfaces that sit exactly on a level.

In [ ]:
!cd {PROJECT}/ml && python -m pytest -q tests/kernels 2>&1 | tee {OUTPUT}/tests.txt | tail -15

## Benchmark

Measures the T4's peak bandwidth with a device-to-device copy, then
times every version at three sizes.

In [ ]:
!cd {PROJECT} && PYTHONPATH=ml/src python scripts/research/run_gpu_benchmark.py --output {OUTPUT}

## Nsight Compute (optional)

Hardware counters are often blocked inside containers. If `ncu` is
missing or blocked, the error is saved instead of a result.

In [ ]:
PROFILE_SCRIPT = (
    "import torch;"
    "from ai_earth_intelligence_and_forecasting_engine.kernels import hdwi_cuda, hdwi_triton;"
    "from ai_earth_intelligence_and_forecasting_engine.kernels.fixtures import random_columns;"
    "c = random_columns((1,), 721, 1440, device='cuda');"
    "hdwi_triton.triton_hot_dry_windy_index(c);"
    "hdwi_cuda.cuda_hot_dry_windy_index(c);"
    "torch.cuda.synchronize()"
)
METRICS = (
    "gpu__time_duration.sum,dram__bytes_read.sum,dram__bytes_write.sum,"
    "dram__throughput.avg.pct_of_peak_sustained_elapsed"
)
!cd {PROJECT} && PYTHONPATH=ml/src ncu --target-processes all -k regex:hdwi --metrics {METRICS} python -c "{PROFILE_SCRIPT}" > {OUTPUT}/ncu.txt 2>&1; tail -40 {OUTPUT}/ncu.txt

In [ ]:
shutil.make_archive("/kaggle/working/gpu_perf", "zip", OUTPUT)
sorted(path.name for path in OUTPUT.rglob("*"))